In [1]:
from os import chdir
from pathlib import Path

cwd = Path.cwd()
print(f"CWD: {cwd}")
if cwd.name == "code":
    chdir("..")
print(f"CWD: {Path.cwd()}")

CWD: /home/james/git/fed-learning/code
CWD: /home/james/git/fed-learning


In [2]:
#Make data sample iterable
from torch.utils.data import Dataset, DataLoader, TensorDataset
import torch
from tqdm import tqdm

import pandas as pd
import numpy as np

import torch.nn as nn
import torch.nn.functional as F

from collections import Counter
import networkx as nx
from networkx.generators.classic import empty_graph
from networkx.utils import discrete_sequence, py_random_state, weighted_choice

In [3]:
# train_df = pd.read_csv("dataset/ml100k_train.csv")
# val_df = pd.read_csv("dataset/ml100k_test.csv")
# n_users = train_df['customer_id'].nunique()
# n_items = train_df['product_code'].nunique()
# train_df.head()

In [4]:
train_df = pd.read_csv("dataset/hm_train_df.csv")
val_df = pd.read_csv("dataset/hm_val_df.csv")
n_users = train_df['customer_id'].nunique()
n_items = train_df['product_code'].nunique()
train_df.head()

,customer_id,product_code,bought
0,0,0,2
1,0,1,2
2,0,2,1
3,0,3,1
4,0,4,1


In [5]:
X_train = train_df[["customer_id", "product_code"]]
X_train.columns = ["user_id", "item_id"]
y_train = train_df["bought"]

X_test = val_df[["customer_id", "product_code"]]
X_test.columns = ["user_id", "item_id"]
y_test = val_df["bought"]
# y_train.columns = [""]

In [6]:
class HMDataset(Dataset):
    def __init__(self, x,y):
        self.x = x.reset_index(drop=True)
        self.y = y.reset_index(drop=True)

    def __len__(self):
        return self.x.shape[0]

    def __getitem__(self, index):

        row = self.x.iloc[index]
        user = torch.tensor(row.user_id).int()
        movie = torch.tensor(row.item_id).int()
        target = torch.tensor(self.y.iloc[index]).float()

        return user,movie, target

HMDataset(X_train,y_train)[0]

(tensor(0, dtype=torch.int32), tensor(0, dtype=torch.int32), tensor(2.))

In [7]:
#Optimizer selection
def get_optimizer(net):
    optimizer = torch.optim.SGD(net.parameters(), lr=0.01)

    return optimizer

In [8]:


#SVD model architecture

# class HMModel(nn.Module):
#     def __init__(self,n_users,n_items):
#         super(HMModel, self).__init__()
#         torch.manual_seed(1)

#         self.user_emb = nn.Embedding(n_users, embedding_dim=30).float()
#         self.article_emb = nn.Embedding(n_items, embedding_dim=30).float()
#         self.user_bias = nn.Embedding(n_users, embedding_dim=1).float()
#         self.article_bias = nn.Embedding(n_items, embedding_dim=1).float()


#     def forward(self, inputs):
#         user_index, item_index = inputs[0], inputs[1]
#         u_e = self.user_emb(user_index)
#         i_e = self.article_emb(item_index)
#         u_b = self.user_bias(user_index)
#         i_b = self.article_bias(item_index)

#         x = (u_e * i_e).sum(1) + u_b + i_b

#         return x




#Generalized SVD model architecture

class HMModel(nn.Module):
    def __init__(self,n_users,n_items):
        super(HMModel, self).__init__()
        torch.manual_seed(1)

        self.user_emb = nn.Embedding(n_users, embedding_dim=30).float()
        self.article_emb = nn.Embedding(n_items, embedding_dim=30).float()
        self.user_bias = nn.Embedding(n_users, embedding_dim=1).float()
        self.article_bias = nn.Embedding(n_items, embedding_dim=1).float()

        self.user_layer_dict =  nn.ModuleDict()
        for i in range(n_users):
          self.user_layer_dict["layer{0}".format(i)] = nn.Linear(30,10)
        self.article_layer_dict = nn.ModuleDict()
        for i in range(n_items):
          self.article_layer_dict["layer{0}".format(i)] = nn.Linear(30,10)


    def forward(self, inputs):
        user_index, item_index = inputs[0], inputs[1]
        u_e = self.user_emb(user_index)
        i_e = self.article_emb(item_index)
        u_e = self.user_layer_dict["layer{0}".format(int(user_index))](u_e)
        i_e = self.article_layer_dict["layer{0}".format(int(item_index))](i_e)

        u_b = self.user_bias(user_index)
        i_b = self.article_bias(item_index)
        x = (u_e * i_e).sum(1) + u_b + i_b

        return x


In [9]:
#Models Initialization
model_dict = {}
for x in tqdm(range(n_users)):
    model_dict["model{0}".format(x)] = HMModel(n_users,n_items)

 14%|██████████████████████████▉                                                                                                                                                                    | 248/1760 [01:15<07:37,  3.30it/s]


KeyboardInterrupt: 

In [ ]:
import sys

def read_data(data):
    return tuple(d for d in data[:-1]), data[-1]

def validate(model_dict, val_loader):

    tbar = val_loader
    criterion = nn.MSELoss()
    loss_list = []
    for i in range(len(model_dict)):
        model_dict["model{0}".format(i)].eval()

    with torch.no_grad():
        for idx, data in enumerate(tbar):
            inputs, target = read_data(data)
            logits = model_dict["model{0}".format(int(inputs[0]))](inputs)
            loss = criterion(logits, target)

            loss_list.append(loss.detach().cpu().item())
        avg_loss = np.sqrt(np.mean(loss_list))

    return avg_loss

val_dataset = HMDataset(X_test,y_test)
val_loader = DataLoader(val_dataset, batch_size=1, shuffle=False)

In [ ]:
def random_k_out_graph(n, k, alpha, self_loops=False, seed=None):
    if alpha < 0:
        raise ValueError("alpha must be positive")
    G = nx.empty_graph(n, create_using=nx.MultiDiGraph)
    weights = Counter({v: alpha for v in G})
    for i in range(k * n):
        u = np.random.choice([v for v, d in G.out_degree() if d < k])

        if not self_loops:
            adjustment = Counter({u: weights[u]})
        else:
            adjustment = Counter()
        v = weighted_choice(weights - adjustment, seed=seed)
        G.add_edge(u, v)
        weights[v] += 1
    return G

#Random_5_out Graph
pairs = []
Graph = random_k_out_graph(n_users,5,50)

In [13]:
from time import perf_counter

In [14]:
sstart = perf_counter()
print(sstart)

120683.881885586


In [15]:
val_loss = []
optimizer_dict = {}
def train(model_dict, train_loader, epochs):

    
    criterion = nn.MSELoss()
    val_map = validate(model_dict, val_loader)
    val_loss.append(val_map)

    #Initialize Optimizers
    for i in tqdm(range(n_users)):
        optimizer_dict["model{0}".format(i)] = get_optimizer(model_dict["model{0}".format(i)])

    for e in range(epochs):
        tbar = tqdm(train_loader, file=sys.stdout)
        for i in range(len(model_dict)):
            model_dict["model{0}".format(i)].train()
        loss_list = []
        for idx, data in enumerate(tbar):
            inputs, target = read_data(data)
            optimizer = optimizer_dict["model{0}".format(int(inputs[0]))]
            optimizer.zero_grad()
            logits = model_dict["model{0}".format(int(inputs[0]))](inputs)
            loss = criterion(logits, target)
            loss.backward()#Calculate Gradients

            #Get gradients of current Model
            model_gradient = {}#Store current gradients
            for name,param in model_dict["model{0}".format(int(inputs[0]))].named_parameters():
                gradient = param.grad
                model_gradient[name] = gradient

            #Assign gradients to neighbors' Model
            user_n = Graph.adj[int(inputs[0])]
            # user_n = list(Adjacency_List[int(inputs[0])])
            #user_n = []  #uncomment it for local learning
            for neighbor in user_n:
                neighbor_model = model_dict["model{0}".format(neighbor)]
                neighbor_optimizer = optimizer_dict["model{0}".format(neighbor)]
                neighbor_model.zero_grad()
                for name,param in neighbor_model.named_parameters():
                    param.grad = model_gradient[name] #Assign gradients to the neighbors
                neighbor_optimizer.step()#Neighbors' update

            optimizer.step()#Current user's update
            loss_list.append(loss.detach().cpu().item())
            avg_loss = np.mean(loss_list)
            tbar.set_description(f"Epoch {e+1} Loss: {np.sqrt(np.round(avg_loss,7))} ")

        val_map = validate(model_dict, val_loader)
        val_loss.append(val_map)
        log_text = f"Epoch {e+1}\nTrain Loss: {np.sqrt(avg_loss)}\nValidation Loss: {val_map}\n"
        print(log_text)



train_dataset = HMDataset(X_train,y_train)
start = perf_counter()
train_loader = DataLoader(train_dataset, batch_size=1, shuffle=True)
train(model_dict, train_loader,epochs=2)
end = perf_counter() - start
print(end)

/home/james/git/fed-learning/.venv/lib/python3.10/site-packages/torch/nn/modules/loss.py:535: UserWarning: Using a target size (torch.Size([1])) that is different to the input size (torch.Size([1, 1])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)
100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1760/1760 [00:00<00:00, 7780.05it/s]


Epoch 1 Loss: 6.3417311043594395 :   0%|                                                                                                                                                           | 37/86480 [00:00<03:59, 361.35it/s]

/home/james/git/fed-learning/.venv/lib/python3.10/site-packages/torch/nn/modules/loss.py:535: UserWarning: Using a target size (torch.Size([1])) that is different to the input size (torch.Size([1, 1])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)


Epoch 1 Loss: 4.41529211944125 : 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 86480/86480 [04:40<00:00, 307.98it/s]
Epoch 1
Train Loss: 4.415292120890613
Validation Loss: 3.4683799485424913

Epoch 2 Loss: 1.703024720901021 : 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 86480/86480 [04:31<00:00, 319.09it/s]
Epoch 2
Train Loss: 1.7030247223776933
Validation Loss: 3.1637680666885672

562.708738226007


In [16]:
final_end = perf_counter()
print(final_end - sstart)

564.7557486120058


In [1]:
val_loss #Evaluation

NameError: name 'val_loss' is not defined

In [19]:
train_dataset = HMDataset(X_train,y_train)
train_loader = DataLoader(train_dataset, batch_size=100, shuffle=True)

In [28]:
train_tensor_dataset = TensorDataset(torch.tensor(train_df.to_numpy()))
train_tensor_loader = DataLoader(train_tensor_dataset, batch_size=100, shuffle=True)

In [55]:
def hm_time_dataloaders(dataset, bs):
    dl = DataLoader(dataset, batch_size=bs, shuffle=True)
    for batch in dl:
        x, y = read_data(batch)

def tensor_time_dataloader(dataset, bs):
    dl = DataLoader(dataset, batch_size=bs, shuffle=True)
    for batch_idx, (x, y) in enumerate(dl):
        x = x
        y = y
        

In [44]:
%timeit hm_time_dataloaders(train_dataset, 1)

15.1 s ± 648 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [56]:
%timeit tensor_time_dataloader(train_tensor_dataset, 1)

4.7 s ± 110 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [20]:
for u, i, r in train_loader:
    print(u, i, r)
    print(type(u))
    break

tensor([ 791, 1512,  647,  105,  531,   33,  889, 1213,  505, 1663,  651, 1286,
        1358,   26, 1540,  942, 1497,  271,  892, 1309,  111, 1416, 1109,  348,
          86,  768, 1487,  502,  786,  981,  174,  114,  946,   21,  152,  530,
        1563,  885,  749,  691, 1008,  177,  620,   69, 1744, 1629,  522,  539,
        1348,  342,  902, 1623,  866,  634,  379, 1582, 1328,   71, 1747,  200,
         267, 1348, 1488, 1237,   53,  188, 1530,  694,  748,  596,  263,  965,
         610,  288, 1108, 1260, 1682,  346, 1134,  895,  118,   43, 1266,   49,
        1715, 1171,  393,  891,  620, 1365,  233,  964,  295, 1168,  814, 1133,
        1012,  634,  578,  132], dtype=torch.int32) tensor([2614,  763, 2038,   42,   92, 1184,  764, 2072, 5840, 1825, 6443, 7599,
        4027,   14,  172, 5417, 1127, 2933,   42, 1572,  137,  240, 2287,  375,
        1902,  651, 5984, 1153, 6902,  853,  163, 2035, 5375,  804, 3230, 5967,
        2735, 1842, 2185, 1132, 6905, 3500, 3510, 1466, 5021, 8472, 

In [48]:
train_tensor_dataset = TensorDataset(torch.tensor(X_train.to_numpy()), torch.tensor(y_train.to_numpy()))
train_tensor_loader = DataLoader(train_tensor_dataset, batch_size=1, shuffle=True)

In [50]:
for x, y in train_tensor_loader:
    print(x)
    print(y)
    print(type(x))
    break

tensor([[1328,  667]])
tensor([1])
<class 'torch.Tensor'>
